# Notebook 01 : ingestion API → MongoDB et simulation des paiements

TP2 BigData Commerce Analytics (B2 Sup de Vinci).

Ce notebook alimente la base MongoDB `commerce` : les 8 ressources de l'API DummyJSON, le fichier des commandes, puis une simulation de paiement pour chaque panier. Il peut être relancé autant de fois que nécessaire, chaque collection est vidée avant d'être rechargée.

## Imports

On charge les bibliothèques utilisées dans tout le notebook :
- `math` pour repérer les valeurs NaN, `random` pour le tirage des statuts de paiement, `datetime` pour noter la date de collecte, `Path` pour manipuler les chemins de fichiers ;
- `pandas` pour lire le CSV et afficher les tableaux de contrôle ;
- `requests` pour appeler l'API DummyJSON ;
- `MongoClient` (bibliothèque pymongo) pour se connecter à MongoDB depuis Python.

In [13]:
import math
import random
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from pymongo import MongoClient

## Repérage des données

Le notebook peut être lancé depuis la racine du projet ou depuis `notebooks/`. On remonte donc les dossiers jusqu'à trouver celui qui contient `data`, et on s'arrête avec un message clair s'il est introuvable.

Dans le code :
- `Path.cwd()` donne le dossier courant, la boucle `while` remonte d'un niveau (`racine.parent`) tant qu'il n'y a pas de dossier `data` ;
- on construit ensuite le chemin du fichier des commandes et on vérifie qu'il existe avant d'aller plus loin.

In [14]:
racine = Path.cwd()
while racine != racine.parent and not (racine / "data").exists():
    racine = racine.parent

if not (racine / "data").exists():
    raise FileNotFoundError("Le dossier data est introuvable.")

dossier_data = racine / "data"
CSV_COMMANDES = dossier_data / "processed" / "commandes_ecommerce.csv"

if not CSV_COMMANDES.exists():
    raise FileNotFoundError(f"Fichier des commandes introuvable : {CSV_COMMANDES}")

print("Dossier du projet :", racine)
print("Fichier des commandes :", CSV_COMMANDES.relative_to(racine))

Dossier du projet : c:\Users\Timéo\Documents\Sup 2 vinci\B2\Big Data\BigData_Commerce_Analytics_DummyJSON
Fichier des commandes : data\processed\commandes_ecommerce.csv


On utilise volontairement `data/processed/commandes_ecommerce.csv`, le fichier indiqué par le TP (800 lignes, 16 colonnes), et pas le premier CSV trouvé dans `data/` : selon le poste, ça pouvait être un autre fichier.

## Étape 5 : connexion à MongoDB

On vérifie que le serveur local répond, puis on sélectionne la base `commerce`.

Dans le code :
- `MongoClient(...)` ouvre la connexion vers le MongoDB qui tourne dans Docker sur le port 27017. `serverSelectionTimeoutMS=3000` fait échouer la cellule au bout de 3 secondes si le serveur ne répond pas, au lieu d'attendre 30 secondes ;
- `ping` est la commande standard pour tester le serveur, elle doit renvoyer `{'ok': 1.0}` ;
- `client["commerce"]` sélectionne la base. MongoDB ne la crée vraiment qu'au premier document inséré, c'est pour ça que la liste des collections peut être vide au premier lancement.

In [15]:
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=3000)

print(client.admin.command("ping"))
print("Version MongoDB :", client.server_info()["version"])

db = client["commerce"]
print("Base sélectionnée :", db.name)
print("Collections existantes :", db.list_collection_names())

{'ok': 1.0}
Version MongoDB : 7.0.48
Base sélectionnée : commerce
Collections existantes : ['raw_todos', 'raw_commandes', 'raw_users', 'raw_products', 'raw_posts', 'raw_quotes', 'raw_payments', 'raw_recipes', 'raw_comments', 'raw_carts']


## Atelier 1 (1/2) : ingestion des ressources API

La fonction `recuperer_ressource(nom)` télécharge une ressource DummyJSON complète.

Dans le code :
- `requests.get(...)` appelle par exemple `https://dummyjson.com/products?limit=0`. Le paramètre `limit=0` demande tous les éléments, sinon l'API n'en renvoie que 30 ;
- `raise_for_status()` arrête le notebook avec une erreur claire si l'API répond autre chose que 200 (page introuvable, serveur en panne...) ;
- la réponse JSON a la forme `{"products": [...], "total": 194, "skip": 0, "limit": 194}`. On vérifie que la clé attendue est bien là, puis on renvoie la liste des documents et le total annoncé.

In [16]:
BASE_URL = "https://dummyjson.com"


def recuperer_ressource(nom):
    response = requests.get(f"{BASE_URL}/{nom}", params={"limit": 0}, timeout=30)
    response.raise_for_status()
    donnees = response.json()

    if nom not in donnees:
        raise ValueError(f"Format inattendu pour la ressource {nom}")

    return donnees[nom], donnees["total"]

Chaque ressource va dans sa collection `raw_<ressource>`. Le préfixe `raw_` indique qu'on stocke les données brutes, telles que l'API les renvoie.

Dans le code :
- le dictionnaire `correspondance` associe chaque ressource de l'API à sa collection MongoDB ;
- pour chaque ressource, on télécharge les documents, on vide la collection (`delete_many({})`) puis on insère la liste (`insert_many`). Vider avant d'insérer permet de relancer la cellule sans créer de doublons ;
- `count_documents({})` compte ce qui est réellement stocké. On range les trois chiffres (total API, reçus, stockés) dans une liste pour afficher un tableau récapitulatif à la fin.

In [17]:
correspondance = {
    "products": "raw_products",
    "carts": "raw_carts",
    "users": "raw_users",
    "posts": "raw_posts",
    "comments": "raw_comments",
    "recipes": "raw_recipes",
    "quotes": "raw_quotes",
    "todos": "raw_todos",
}

date_collecte = datetime.now()
print("Date de collecte :", date_collecte.strftime("%d/%m/%Y %H:%M:%S"))

recap = []
for ressource, collection_nom in correspondance.items():
    documents, total_api = recuperer_ressource(ressource)

    collection = db[collection_nom]
    collection.delete_many({})
    if documents:
        collection.insert_many(documents)

    nb_mongo = collection.count_documents({})
    recap.append({
        "ressource": ressource,
        "collection": collection_nom,
        "total_api": total_api,
        "documents_recus": len(documents),
        "documents_mongo": nb_mongo,
        "ok": total_api == len(documents) == nb_mongo,
    })

df_recap = pd.DataFrame(recap)
display(df_recap)
print("Total documents insérés :", df_recap["documents_mongo"].sum())

Date de collecte : 26/11/2026 13:35:56


,ressource,collection,total_api,documents_recus,documents_mongo,ok
0,products,raw_products,194,194,194,True
1,carts,raw_carts,208,208,208,True
2,users,raw_users,208,208,208,True
3,posts,raw_posts,251,251,251,True
4,comments,raw_comments,340,340,340,True
5,recipes,raw_recipes,50,50,50,True
6,quotes,raw_quotes,1454,1454,1454,True
7,todos,raw_todos,254,254,254,True


Total documents insérés : 2959


On compare trois chiffres pour chaque ressource : ce que l'API annonce (`total`), ce qu'on a reçu, et ce qui est réellement stocké dans MongoDB. Résultat attendu : 2 959 documents au total, `ok` partout.

Exemple de document : `find_one` renvoie un seul document de la collection. Le deuxième argument choisit les champs à afficher (1 = afficher) pour que le résultat reste lisible. On voit que MongoDB a ajouté un champ `_id` (identifiant technique, de type ObjectId) à côté du champ `id` venu de DummyJSON.

In [18]:
db.raw_products.find_one({}, {"_id": 1, "id": 1, "title": 1, "category": 1, "price": 1, "stock": 1})

{'_id': ObjectId('6b0827ac4c0cb0e73494e31c'),
 'id': 1,
 'title': 'Essence Mascara Lash Princess',
 'category': 'beauty',
 'price': 9.99,
 'stock': 99}

## Atelier 1 (2/2) : import des commandes et simulation des paiements

### 1. Import du fichier des commandes

Une ligne du fichier correspond à un produit dans un panier. Le fichier est importé tel quel dans `raw_commandes`, sans supprimer de ligne.

Dans le code :
- `pd.read_csv` lit le fichier, puis `pd.to_datetime` transforme `order_date` en vraie date (sinon MongoDB la stockerait comme du texte) ;
- `to_dict(orient="records")` transforme le tableau en liste de dictionnaires : un dictionnaire par ligne, donc un document MongoDB par ligne ;
- la double boucle remplace les NaN (valeur manquante côté pandas, surtout dans `brand`) par `None`, qui devient `null` dans MongoDB. Sans ça, MongoDB stockerait un nombre « NaN » dans un champ texte ;
- comme pour les ressources API, on vide la collection avant d'insérer.

In [19]:
df_commandes = pd.read_csv(CSV_COMMANDES)
df_commandes["order_date"] = pd.to_datetime(df_commandes["order_date"])
print("Dimensions :", df_commandes.shape)

documents_commandes = df_commandes.to_dict(orient="records")
for doc in documents_commandes:
    for champ, valeur in doc.items():
        if isinstance(valeur, float) and math.isnan(valeur):
            doc[champ] = None

collection_commandes = db["raw_commandes"]
collection_commandes.delete_many({})
collection_commandes.insert_many(documents_commandes)

Dimensions : (800, 16)


InsertManyResult([ObjectId('6b0827b24c0cb0e73494eeab'), ObjectId('6b0827b24c0cb0e73494eeac'), ObjectId('6b0827b24c0cb0e73494eead'), ObjectId('6b0827b24c0cb0e73494eeae'), ObjectId('6b0827b24c0cb0e73494eeaf'), ObjectId('6b0827b24c0cb0e73494eeb0'), ObjectId('6b0827b24c0cb0e73494eeb1'), ObjectId('6b0827b24c0cb0e73494eeb2'), ObjectId('6b0827b24c0cb0e73494eeb3'), ObjectId('6b0827b24c0cb0e73494eeb4'), ObjectId('6b0827b24c0cb0e73494eeb5'), ObjectId('6b0827b24c0cb0e73494eeb6'), ObjectId('6b0827b24c0cb0e73494eeb7'), ObjectId('6b0827b24c0cb0e73494eeb8'), ObjectId('6b0827b24c0cb0e73494eeb9'), ObjectId('6b0827b24c0cb0e73494eeba'), ObjectId('6b0827b24c0cb0e73494eebb'), ObjectId('6b0827b24c0cb0e73494eebc'), ObjectId('6b0827b24c0cb0e73494eebd'), ObjectId('6b0827b24c0cb0e73494eebe'), ObjectId('6b0827b24c0cb0e73494eebf'), ObjectId('6b0827b24c0cb0e73494eec0'), ObjectId('6b0827b24c0cb0e73494eec1'), ObjectId('6b0827b24c0cb0e73494eec2'), ObjectId('6b0827b24c0cb0e73494eec3'), ObjectId('6b0827b24c0cb0e73494ee

Contrôle de l'import : on compare le CSV (côté pandas) et MongoDB sur quatre points.

Dans le code :
- `aggregate` avec `$group` et `_id: None` regroupe tous les documents en un seul groupe et fait la somme de `discounted_line_total` directement dans MongoDB ;
- `distinct("cart_id")` renvoie la liste des paniers différents, on compte sa longueur ;
- `count_documents({"brand": None})` compte les documents dont la marque est `null` ;
- la colonne `ok` compare les deux valeurs. Résultat attendu : 800 documents, 208 paniers, 3 456 709,58 et 377 lignes sans marque.

In [20]:
somme_mongo = list(collection_commandes.aggregate([
    {"$group": {"_id": None, "total": {"$sum": "$discounted_line_total"}}}
]))[0]["total"]

controles = pd.DataFrame([
    {"contrôle": "Nombre de documents", "CSV": len(df_commandes),
     "MongoDB": collection_commandes.count_documents({})},
    {"contrôle": "Paniers distincts (cart_id)", "CSV": df_commandes["cart_id"].nunique(),
     "MongoDB": len(collection_commandes.distinct("cart_id"))},
    {"contrôle": "Somme discounted_line_total", "CSV": round(df_commandes["discounted_line_total"].sum(), 2),
     "MongoDB": round(somme_mongo, 2)},
    {"contrôle": "Lignes sans marque (brand)", "CSV": int(df_commandes["brand"].isna().sum()),
     "MongoDB": collection_commandes.count_documents({"brand": None})},
])
controles["ok"] = controles["CSV"] == controles["MongoDB"]
display(controles)

,contrôle,CSV,MongoDB,ok
0,Nombre de documents,800.00,800.00,True
1,Paniers distincts (cart_id),208.00,208.00,True
2,Somme discounted_line_total,3456709.58,3456709.58,True
3,Lignes sans marque (brand),377.00,377.00,True


### 2. Simulation des paiements

**Contrôle de cohérence par panier.** Avant de créer les paiements, on vérifie que chaque panier a un seul client, un seul montant total et une seule commande. Sinon on ne saurait pas qui facturer ni combien.

Dans le code : `groupby("cart_id")` regroupe les lignes par panier et `nunique` compte les valeurs différentes dans chaque groupe. On garde ensuite les paniers où l'un des compteurs est différent de 1. On doit en trouver 0.

In [21]:
controle_paniers = (
    df_commandes
    .groupby("cart_id")
    .agg(
        nombre_clients=("user_id", "nunique"),
        nombre_montants=("cart_discounted_total", "nunique"),
        nombre_commandes=("order_id", "nunique"),
    )
)

paniers_incoherents = controle_paniers[
    (controle_paniers["nombre_clients"] != 1)
    | (controle_paniers["nombre_montants"] != 1)
    | (controle_paniers["nombre_commandes"] != 1)
]
print("Paniers :", len(controle_paniers))
print("Paniers incohérents :", len(paniers_incoherents))

Paniers : 208
Paniers incohérents : 0


**Comment on génère les paiements**

- **Un paiement par panier.** On regroupe les 800 lignes de `raw_commandes` par `cart_id` (agrégation MongoDB), ce qui donne 208 paniers. Faire un paiement par ligne reviendrait à facturer plusieurs fois le même panier.
- **Éléments repris du panier :** `cart_id` et `order_id` pour la commande, `user_id` pour le client, `cart_discounted_total` pour le montant (ce que le client paie après remise). Ce total est répété sur chaque ligne du panier, on ne le prend donc qu'une fois (`$first`) au lieu de l'additionner.
- **Statut :** tirage pondéré avec `random.choices`, 70 % PAID, 20 % REFUSED, 10 % PENDING. La graine 42 rend le tirage reproductible : mêmes résultats à chaque exécution et sur nos deux postes.
- **Date de paiement :** la date de la première ligne du panier. Attention, `order_date` est générée pour l'exercice (une date au hasard par ligne), cette date n'a pas de valeur réelle. On avait d'abord pris la date la plus récente du panier, mais le maximum de plusieurs dates tirées au hasard tombe presque toujours en fin d'année : ça créait une fausse hausse des ventes en fin d'année dans l'indicateur mensuel.
- **Unicité :** un index unique sur `cart_id` empêche MongoDB d'accepter un deuxième paiement pour le même panier.

Dans le code :
- l'agrégation `$group` sur `$cart_id` produit un document par panier. Le premier `$sort` sur `_id` remet les lignes dans leur ordre d'insertion pour que « la première ligne » soit toujours la même. `$first` prend la valeur de la première ligne du panier (pour le client et le montant, elle est identique sur toutes les lignes), `$sum` additionne. Le second `$sort` trie les paniers par numéro pour que le tirage soit toujours fait dans le même ordre ;
- `random.Random(42)` crée un générateur aléatoire avec la graine 42, et `rng.choices(STATUTS, weights=POIDS)` tire un statut en respectant les poids 70 / 20 / 10 ;
- pour chaque panier on construit le document de paiement. `amount_collected` vaut le montant si le paiement est PAID, 0 sinon ;
- `drop()` supprime l'ancienne collection (et son index), `create_index("cart_id", unique=True)` crée l'index unique, puis `insert_many` insère les 208 paiements.

In [22]:
paniers = list(collection_commandes.aggregate([
    {"$sort": {"_id": 1}},  # ordre d'insertion des lignes
    {"$group": {
        "_id": "$cart_id",
        "order_id": {"$first": "$order_id"},
        "user_id": {"$first": "$user_id"},
        "montant": {"$first": "$cart_discounted_total"},
        "nb_lignes": {"$sum": 1},
        "nb_articles": {"$sum": "$quantity"},
        "date_paiement": {"$first": "$order_date"},
    }},
    {"$sort": {"_id": 1}},  # ordre fixe pour que le tirage soit reproductible
]))
print("Paniers à payer :", len(paniers))

STATUTS = ["PAID", "REFUSED", "PENDING"]
POIDS = [70, 20, 10]
rng = random.Random(42)

paiements = []
for i, panier in enumerate(paniers, start=1):
    statut = rng.choices(STATUTS, weights=POIDS)[0]
    montant = round(panier["montant"], 2)
    paiements.append({
        "payment_id": f"PAY-{i:05d}",
        "cart_id": panier["_id"],
        "order_id": panier["order_id"],
        "user_id": panier["user_id"],
        "amount": montant,
        "status": statut,
        "amount_collected": montant if statut == "PAID" else 0.0,
        "payment_date": panier["date_paiement"],
        "nb_lignes": panier["nb_lignes"],
        "nb_articles": panier["nb_articles"],
    })

collection_paiements = db["raw_payments"]
collection_paiements.drop()
collection_paiements.create_index("cart_id", unique=True)
collection_paiements.insert_many(paiements)
print("Paiements insérés :", collection_paiements.count_documents({}))

Paniers à payer : 208
Paiements insérés : 208


**Contrôles de cohérence des paiements.**

Dans le code :
- on compare l'ensemble des `cart_id` des commandes et celui des paiements : la différence doit être vide ;
- l'agrégation regroupe les paiements par `cart_id` et ne garde (`$match`) que les groupes qui ont plus d'un paiement : on doit n'en trouver aucun ;
- on vérifie que seuls les trois statuts prévus apparaissent ;
- le dernier contrôle est le plus important : si la somme des 208 paiements est égale à la somme des 800 lignes de commandes, aucun montant n'a été perdu ni compté deux fois.

In [23]:
ids_commandes = set(collection_commandes.distinct("cart_id"))
ids_paiements = set(collection_paiements.distinct("cart_id"))
doublons = list(collection_paiements.aggregate([
    {"$group": {"_id": "$cart_id", "n": {"$sum": 1}}},
    {"$match": {"n": {"$gt": 1}}},
]))
total_paiements = sum(p["amount"] for p in collection_paiements.find({}, {"amount": 1}))
statuts_trouves = set(collection_paiements.distinct("status"))

controles_paiements = pd.DataFrame([
    {"contrôle": "Un paiement par panier",
     "valeur": collection_paiements.count_documents({}),
     "ok": collection_paiements.count_documents({}) == len(ids_commandes)},
    {"contrôle": "Aucun cart_id en double",
     "valeur": len(doublons), "ok": len(doublons) == 0},
    {"contrôle": "Tous les paniers ont un paiement",
     "valeur": len(ids_commandes - ids_paiements), "ok": ids_commandes == ids_paiements},
    {"contrôle": "Statuts autorisés uniquement",
     "valeur": str(sorted(statuts_trouves)), "ok": statuts_trouves <= set(STATUTS)},
    {"contrôle": "Somme des paiements = somme des lignes",
     "valeur": round(total_paiements, 2), "ok": round(total_paiements, 2) == round(somme_mongo, 2)},
])
display(controles_paiements)

,contrôle,valeur,ok
0,Un paiement par panier,208,True
1,Aucun cart_id en double,0,True
2,Tous les paniers ont un paiement,0,True
3,Statuts autorisés uniquement,"['PAID', 'PENDING', 'REFUSED']",True
4,Somme des paiements = somme des lignes,3456709.58,True


### 3. Analyse des résultats

Dans le code :
- l'agrégation regroupe les paiements par `status` et calcule pour chaque statut le nombre de paiements, le montant total associé et le montant réellement encaissé (`amount_collected`) ;
- on ajoute ensuite avec pandas les pourcentages (part des paiements, part du montant) et la probabilité théorique pour comparer ;
- le chiffre d'affaires encaissé est la somme de `montant_encaisse`, donc uniquement les PAID. Le taux de refus est le nombre de REFUSED divisé par le nombre total de tentatives.

In [24]:
analyse = list(collection_paiements.aggregate([
    {"$group": {
        "_id": "$status",
        "nb_paiements": {"$sum": 1},
        "montant_total": {"$sum": "$amount"},
        "montant_encaisse": {"$sum": "$amount_collected"},
    }},
    {"$sort": {"nb_paiements": -1}},
]))

df_analyse = pd.DataFrame(analyse).rename(columns={"_id": "statut"})
df_analyse = df_analyse[["statut", "nb_paiements", "montant_total", "montant_encaisse"]]
df_analyse["% paiements"] = (df_analyse["nb_paiements"] / df_analyse["nb_paiements"].sum() * 100).round(1)
df_analyse["% montant"] = (df_analyse["montant_total"] / df_analyse["montant_total"].sum() * 100).round(1)
df_analyse["% théorique"] = df_analyse["statut"].map({"PAID": 70, "REFUSED": 20, "PENDING": 10})
df_analyse[["montant_total", "montant_encaisse"]] = df_analyse[["montant_total", "montant_encaisse"]].round(2)
display(df_analyse)

ca_encaisse = df_analyse["montant_encaisse"].sum()
ca_total = df_analyse["montant_total"].sum()
nb_refus = collection_paiements.count_documents({"status": "REFUSED"})
taux_refus = nb_refus / collection_paiements.count_documents({}) * 100

print(f"Montant total des paniers   : {ca_total:,.2f}")
print(f"Chiffre d'affaires encaissé : {ca_encaisse:,.2f} ({ca_encaisse / ca_total * 100:.1f} % du total)")
print(f"Montant non encaissé        : {ca_total - ca_encaisse:,.2f}")
print(f"Taux de refus               : {taux_refus:.2f} %")

,statut,nb_paiements,montant_total,montant_encaisse,% paiements,% montant,% théorique
0,PAID,150,2669235.61,2669235.61,72.1,77.2,70
1,REFUSED,37,423555.75,0.00,17.8,12.3,20
2,PENDING,21,363918.22,0.00,10.1,10.5,10


Montant total des paniers   : 3,456,709.58
Chiffre d'affaires encaissé : 2,669,235.61 (77.2 % du total)
Montant non encaissé        : 787,473.97
Taux de refus               : 17.79 %


### Commentaire

| Statut | Paiements | Part | Montant associé |
|---|---:|---:|---:|
| PAID | 150 | 72,1 % | 2 669 235,61 |
| REFUSED | 37 | 17,8 % | 423 555,75 |
| PENDING | 21 | 10,1 % | 363 918,22 |
| Total | 208 | 100 % | 3 456 709,58 |

Le chiffre d'affaires encaissé est de 2 669 235,61, soit environ 77 % du montant total des paniers. Seuls les paiements PAID comptent : les paniers REFUSED et PENDING représentent quand même 787 473,97 de commandes, mais rien n'est encaissé dans ce scénario.

Les proportions obtenues (72 / 18 / 10 %) sont proches des probabilités demandées (70 / 20 / 10 %). L'écart vient du hasard du tirage sur seulement 208 paniers, il serait plus faible avec plus de paniers. Le taux de refus est de 17,79 %.

On remarque que les paiements PAID pèsent 77 % du montant alors qu'ils ne représentent que 72 % des paiements : les paniers acceptés sont en moyenne un peu plus chers. Ce n'est pas lié à la règle de simulation (le statut ne dépend pas du montant), c'est le hasard, accentué par quelques paniers très chers de la catégorie vehicle.

Pour être sûrs qu'aucun panier n'est compté deux fois, on a l'index unique sur `cart_id`, la recherche de doublons qui renvoie 0, et la somme des paiements qui retombe exactement sur la somme des lignes de commandes.